# Master Parquet Merge — Swap PrISM + Add UTD-MHAD

1. Load existing master parquet
2. Drop old `prism` rows
3. Concat new PrISM (procedural sub-task labels) + UTD-MHAD
4. Sanity-check row counts per dataset
5. Save

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from pathlib import Path
from collections import Counter

TEMP = Path("/home/aidan/IMU_LM_Data/Unification/temp")

master_path = TEMP / "IMULM_master_dataset.parquet"
prism_path  = TEMP / "prism_clean_data_procedural.parquet"
utd_path    = TEMP / "utd_mhad_clean_data.parquet"
out_path    = TEMP / "IMULM_master_dataset_v2.parquet"

# ---- 1. Inspect master metadata (no full load) ----
pf = pq.ParquetFile(master_path)
print(f"Master: {pf.metadata.num_rows:,} rows, {pf.metadata.num_row_groups} row groups")
print(f"Columns: {pf.schema_arrow.names}")

# ---- 2. Load the two small tables ----
print("\nLoading new PrISM...")
prism_new = pq.read_table(prism_path)
print(f"  PrISM rows: {prism_new.num_rows:,}, subjects: {len(pc.unique(prism_new.column('subject_id')))}, "
      f"sessions: {len(pc.unique(prism_new.column('session_id')))}, "
      f"labels: {len(pc.unique(prism_new.column('dataset_activity_label')))}")

print("Loading UTD-MHAD...")
utd = pq.read_table(utd_path)
print(f"  UTD-MHAD rows: {utd.num_rows:,}, subjects: {len(pc.unique(utd.column('subject_id')))}, "
      f"sessions: {len(pc.unique(utd.column('session_id')))}, "
      f"labels: {len(pc.unique(utd.column('dataset_activity_label')))}")

# ---- 3. Schema check (against master) ----
print("\n=== SCHEMA CHECK ===")
master_schema = pf.schema_arrow
for tbl_name, tbl in [("prism", prism_new), ("utd", utd)]:
    for col in master_schema.names:
        if col not in tbl.schema.names:
            print(f"  MISSING {col} in {tbl_name}")
    for col in tbl.schema.names:
        if col not in master_schema.names:
            print(f"  EXTRA {col} in {tbl_name}")
print("Schema check done (int16→int32 promotion handled automatically).")

# ---- 4. Stream master row-by-row-group, drop prism, write to v2 ----
print(f"\nStreaming master → {out_path.name} (dropping prism rows)...")
writer = None
before_counts = Counter()
rows_written = 0
prism_dropped = 0

for i in range(pf.metadata.num_row_groups):
    rg = pf.read_row_group(i)
    ds_col = rg.column("dataset")
    
    # Count before
    vc = pc.value_counts(ds_col)
    for item in vc.to_pylist():
        before_counts[item["values"]] += item["counts"]
    
    # Filter out prism
    keep = pc.not_equal(ds_col, "prism")
    n_prism = rg.num_rows - pc.sum(keep.cast(pa.int64())).as_py()
    prism_dropped += n_prism
    
    if n_prism < rg.num_rows:  # at least some non-prism rows
        filtered = rg.filter(keep)
        if writer is None:
            writer = pq.ParquetWriter(out_path, filtered.schema)
        writer.write_table(filtered)
        rows_written += filtered.num_rows
    
    if (i + 1) % 10 == 0 or i == pf.metadata.num_row_groups - 1:
        print(f"  Row group {i+1}/{pf.metadata.num_row_groups} — written: {rows_written:,}, prism dropped: {prism_dropped:,}")

# ---- 5. Append new PrISM + UTD-MHAD ----
print("\nAppending new PrISM...")
# Cast int16 → int32 to match master schema
for col in ["global_activity_id", "dataset_activity_id"]:
    if prism_new.schema.field(col).type != pa.int32():
        prism_new = prism_new.set_column(
            prism_new.schema.get_field_index(col), col,
            prism_new.column(col).cast(pa.int32()))
    if utd.schema.field(col).type != pa.int32():
        utd = utd.set_column(
            utd.schema.get_field_index(col), col,
            utd.column(col).cast(pa.int32()))

writer.write_table(prism_new)
rows_written += prism_new.num_rows
print(f"  +{prism_new.num_rows:,} PrISM rows")

print("Appending UTD-MHAD...")
writer.write_table(utd)
rows_written += utd.num_rows
print(f"  +{utd.num_rows:,} UTD-MHAD rows")

writer.close()
print(f"\nDone! Wrote {rows_written:,} total rows to {out_path.name}")

# ---- 6. Summary ----
print("\n=== BEFORE (master) ===")
for ds in sorted(before_counts):
    print(f"  {ds:20s} {before_counts[ds]:>12,}")
print(f"  {'TOTAL':20s} {sum(before_counts.values()):>12,}")

after_counts = dict(before_counts)
after_counts.pop("prism", None)
after_counts["prism"] = prism_new.num_rows
after_counts["utd_mhad"] = after_counts.get("utd_mhad", 0) + utd.num_rows

print("\n=== AFTER (v2) ===")
for ds in sorted(after_counts):
    print(f"  {ds:20s} {after_counts[ds]:>12,}")
print(f"  {'TOTAL':20s} {rows_written:>12,}")
print(f"\nTotal datasets: {len(after_counts)}")
print(f"File size: {out_path.stat().st_size / 1e9:.2f} GB")

Loading master (Arrow)...
  Master rows: 734,080,597
  Master cols: ['dataset', 'subject_id', 'session_id', 'timestamp_ns', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'global_activity_id', 'global_activity_label', 'dataset_activity_id', 'dataset_activity_label']

Loading new PrISM (Arrow)...
  PrISM rows: 2,928,148
Loading UTD-MHAD (Arrow)...
  UTD-MHAD rows: 119,897

=== SCHEMA CHECK ===
Column names: MATCH
  global_activity_id: master=int32, prism=int16, utd=int16  << MISMATCH
  dataset_activity_id: master=int32, prism=int16, utd=int16  << MISMATCH
Dtype check done.

Before — rows per dataset:
  opportunity++             985,421
  pamap2                  1,271,041
  recofit                 7,745,018
  samosa                  1,204,655
  ut_watch                8,146,709
  wear                    3,462,973
  capture24             699,011,946
  prism                   3,037,272
  shoaib                  1,170,000
  wisdm                   8,045,562

Dropping 3,037,272 old

: 

In [ ]:
# ---- Optional: verify the output file ----
pf_out = pq.ParquetFile(out_path)
print(f"Output: {pf_out.metadata.num_rows:,} rows, {pf_out.metadata.num_row_groups} row groups")
print(f"Schema: {pf_out.schema_arrow}")